# Module 2 · Notebook 1 — Vectors & Transformations
### Computing, Robotics & Bionics · companion to A. Géron, *Hands-On Machine Learning with Scikit-Learn and PyTorch*

A **self-contained** geometric linear-algebra reference (no other notebook needed). It covers the vector
and matrix material of Géron's `math_linear_algebra`, re-ordered around **geometry** and visualised live:
vector spaces, linear independence and basis; the scalar product and projection; norms and what counts as
a length; matrices as maps; the transformation zoo (scaling, rotation, reflection, shear, projection);
orthonormal matrices and rigid motions; the determinant and rank; change of basis in the $\Gamma_{BE}$
notation and the conjugation trick that uses it (rotations,
reflections and projections built in an easy basis); and vector spaces beyond $\mathbb R^n$. The rotations and orthonormal frames here are the same
$SO(3)$ ideas as the Robotics & Bionics unit.

Run in **Google Colab** (*Runtime → Run all*). Everything is offline, except the optional bonus
image download at the end (falls back to a placeholder if you have no internet access).

**Contents**
1. Vector spaces, linear independence, span and basis
2. The scalar product in a general basis
3. Orthonormal bases and the dot product formula
4. Norms: what counts as a length
5. Geometric meaning: angle, orthogonality, cosine similarity
6. Projection
7. Gram-Schmidt orthonormalization
8. Matrices as maps (columns = images of the basis)
9. The transformation zoo (visualised)
10. Composition and non-commutativity
11. Orthogonal matrices and rigid motions
12. The determinant as area scaling, and rank
13. Change of basis: the $\Gamma_{BE}$ notation, and the conjugation trick
14. Vector spaces beyond $\mathbb R^n$: polynomials and matrices

Bonus: uncovering a hidden image (Holbein's *The Ambassadors*)

Every section ends with an **Exercise**; run the **Solution** cell to check.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.set_printoptions(precision=3, suppress=True)
print("ready")

---
## 1 · Vector spaces, linear independence, span and basis

Before any coordinates, the object itself. A **vector space** is a set with an addition and a
scalar multiplication obeying ten axioms — closure, commutativity, associativity, a zero vector,
additive inverses, and the four that tie scaling to addition. Nothing in that list mentions arrays
of numbers, which is exactly why $2\times2$ matrices and polynomials qualify too (Section 14).

Inside a space, a **linear combination** of $\mathbf v_1,\dots,\mathbf v_k$ is any
$c_1\mathbf v_1 + \cdots + c_k\mathbf v_k$, and their **span** is the set of all of them. The set is
**linearly independent** when the only combination equal to $\mathbf 0$ is the trivial one — no
vector in it is redundant. A **basis** is an independent set that spans the space, and every basis
of a given space has the same size, its **dimension**.

In practice nobody checks the definition by hand: the **rank** of the matrix holding the vectors as
columns counts how many of them are independent. For biosignals that count has a physical meaning
— wire three electrodes so that one lead is the difference of the other two and you record a
rank-2 cloud, no matter how long you record for. The missing direction was never there to find.


In [ ]:
# Four leads from a 3-electrode montage: lead 4 is wired as lead 1 - lead 2,
# so it carries nothing new. Independence is a rank question.
leads = np.array([
    [1.0, 0.0, 0.5,  1.0],        # each column is one lead's response to a source
    [0.0, 1.0, 0.5, -1.0],
    [0.0, 0.0, 1.0,  0.0],
])
print("4 leads measured in a 3-dimensional space -> at most 3 can be independent")
print("rank of all four columns :", np.linalg.matrix_rank(leads))
print("rank of the first three  :", np.linalg.matrix_rank(leads[:, :3]))

# the explicit dependence: lead 4 = lead 1 - lead 2
coeffs, *_ = np.linalg.lstsq(leads[:, :3], leads[:, 3], rcond=None)
print("lead 4 as a combination of leads 1-3:", coeffs.round(6))
print("reconstruction exact:", np.allclose(leads[:, :3] @ coeffs, leads[:, 3]))

# the first three ARE a basis of R^3: independent, and they span everything
B0 = leads[:, :3]
print("\nbasis of R^3? rank", np.linalg.matrix_rank(B0), "= dimension 3 ->",
      np.linalg.matrix_rank(B0) == 3)
target = np.array([0.4, -1.2, 0.9])
print("so any reading is reachable:", target, "has coordinates",
      np.linalg.solve(B0, target).round(3), "in that basis")


**Exercise 1.** A four-electrode EEG montage gives the response matrix
`M = np.array([[1.0, 2.0, 0.0, 3.0], [0.0, 1.0, 1.0, 2.0], [2.0, 4.0, 0.0, 6.0]])`, one electrode
per column. (a) State the dimension of the space the columns live in, and how many of them can be
independent at most. (b) Compute the rank and say how many electrodes carry genuinely new
information. (c) Write one column as an explicit combination of two others and verify it.
(d) Do any three of the columns form a basis of $\mathbb R^3$? Justify it with a determinant.

> 🤖 *Gemini tip:* "Explain linear independence, span and basis for a set of column vectors, and show me how to use numpy.linalg.matrix_rank and numpy.linalg.lstsq to find the rank of a matrix and express a dependent column as a combination of the others."


In [ ]:
M = np.array([[1.0, 2.0, 0.0, 3.0],
              [0.0, 1.0, 1.0, 2.0],
              [2.0, 4.0, 0.0, 6.0]])

# Your code here

---
## 2 · The scalar product in a general basis

Coordinates are numbers *relative to a chosen basis* `e1, e2, ...`: `v = v1*e1 + v2*e2 + ...`. The **scalar product** `<v,w>` cannot be read off those numbers alone unless you also know how the basis vectors relate to each other -- via the **Gram matrix** `G[i,j] = <e_i, e_j>`, so `<v,w> = v.T @ G @ w`. If the basis is *not* orthonormal, `G` is not the identity, and `<v,w>` is *not* simply `sum(v_i * w_i)`.


In [ ]:
# A basis that is NOT orthonormal
e1 = np.array([1.0, 0.0])
e2 = np.array([1.0, 1.0])
G = np.array([[e1 @ e1, e1 @ e2],
              [e2 @ e1, e2 @ e2]])
print("Gram matrix G:\n", G)

v_coords = np.array([2.0, 1.0])    # v = 2*e1 + 1*e2
w_coords = np.array([1.0, -1.0])   # w = 1*e1 - 1*e2

scalar_via_G = v_coords @ G @ w_coords
v_std = v_coords[0] * e1 + v_coords[1] * e2
w_std = w_coords[0] * e1 + w_coords[1] * e2
scalar_via_dot = v_std @ w_std

print("<v,w> via v.T @ G @ w                  :", scalar_via_G)
print("<v,w> via ordinary dot (standard coords):", scalar_via_dot)


**Exercise 2.** Two EMG electrodes are mounted at a fixed, non-perpendicular angle, so their reading axes form a basis `f1 = np.array([1.0, 0.0])`, `f2 = np.array([0.5, 1.0])` (not orthonormal). Build the Gram matrix `G` for this basis, then for the coordinate vectors `v = (2, -1)` and `w = (1, 1)` (i.e. `v = 2*f1 - f2`, `w = f1 + f2`) compute `<v,w>` two ways: via `v.T @ G @ w`, and via the ordinary dot product after writing `v, w` out in standard coordinates. Confirm the two answers agree.

> 🤖 *Gemini tip:* "Given two non-orthogonal 2-D basis vectors in NumPy, show me how to build their Gram matrix and use it to compute the scalar product of two vectors given in that basis's coordinates, then check it against the ordinary dot product in standard coordinates."


In [ ]:
f1 = np.array([1.0, 0.0])
f2 = np.array([0.5, 1.0])

# Your code here

---
## 3 · Orthonormal bases and the dot product formula

A basis is **orthonormal** when `<e_i, e_j> = 1` if `i == j` else `0` -- exactly when `G = I`. Then `<v,w> = v.T @ w = sum(v_i * w_i)`: the familiar **dot product** is a *consequence* of orthonormality, not a definition in its own right. NumPy's arrays underline the point: a 1-D array has no row/column orientation at all, so `x.T` does nothing and `x @ y` just works -- the row-times-column bookkeeping of `x.T @ y` is a convention for literal matrices, not a property of the scalar product itself.


In [ ]:
# The standard basis is orthonormal: its Gram matrix is the identity
I2 = np.eye(2)
print("standard-basis Gram matrix:\n", I2)

x = np.array([3.0, -2.0])
y = np.array([1.0, 4.0])
print("x @ y                      :", x @ y)   # matches x.T @ I2 @ y trivially
print("x.T is a no-op on a 1-D array:", np.array_equal(x.T, x))


**Exercise 3.** A wearable IMU reports one 3-axis acceleration sample as `x = np.array([1.0, 2.0,
2.0])` (m/s², x/y/z). (a) Confirm `x.T` is exactly the same array (no-op) by checking the shape and
values; (b) compute the dot product two ways -- `x @ x`, and by reshaping to literal row/column
matrices, `x.reshape(1, -1) @ x.reshape(-1, 1)` -- and show they agree once you pull the scalar out
of the 1x1 result. Add a one-line comment on why NumPy's 1-D arrays skip the row/column
bookkeeping, and what that means when you hand the sample to a library expecting a column.

> 🤖 *Gemini tip:* "Explain why a 1-D NumPy array's .T does nothing, how that differs from a 2-D column vector, and show the shapes of x @ x versus x.reshape(1,-1) @ x.reshape(-1,1)."


In [ ]:
x = np.array([1.0, 2.0, 2.0])

# Your code here

---
## 4 · Norms: what counts as a length

"Length" deserves a definition of its own, because a vector space admits more than one and machine
learning uses several. A **norm** is any rule $\lVert\cdot\rVert$ assigning a number to each vector
and obeying three requirements, for all $\mathbf v,\mathbf w$ and every scalar $c$:

1. **Positivity** — $\lVert\mathbf v\rVert \ge 0$, and $\lVert\mathbf v\rVert = 0$ only for $\mathbf v = \mathbf 0$.
2. **Homogeneity** — $\lVert c\,\mathbf v\rVert = |c|\,\lVert\mathbf v\rVert$.
3. **Triangle inequality** — $\lVert\mathbf v + \mathbf w\rVert \le \lVert\mathbf v\rVert + \lVert\mathbf w\rVert$.

The scalar product supplies one for free: $\lVert\mathbf v\rVert = \sqrt{\langle\mathbf v,\mathbf v\rangle}$,
the norm **induced** by the inner product, which in an orthonormal basis is the **Euclidean** or **L2**
norm $\lVert\mathbf x\rVert_2 = \sqrt{x_1^2+\cdots+x_n^2}$. Two others earn their keep in ML and are
read straight off the coordinates: the **L1 norm** $\sum_i|x_i|$ (Lasso) and the **L$\infty$ norm**
$\max_i|x_i|$. All three sit in the family $\lVert\mathbf x\rVert_p = (\sum_i|x_i|^p)^{1/p}$ at
$p = 1, 2$ and the limit $p\to\infty$. A vector of length one is a **unit vector**, and dividing by
the norm is called **normalising**.


In [ ]:
x = np.array([3.0, 4.0])
print("L2 norm :", np.linalg.norm(x))         # 5.0
print("L1 norm :", np.linalg.norm(x, 1))      # 7.0
print("Linf    :", np.linalg.norm(x, np.inf)) # 4.0
print("unit    :", x / np.linalg.norm(x))     # length-1 version

Only the **L2** norm comes from an inner product, and that one fact decides what geometry is
available. A norm is induced by some scalar product exactly when it obeys the **parallelogram law**

$$\lVert\mathbf v+\mathbf w\rVert^2 + \lVert\mathbf v-\mathbf w\rVert^2 \;=\; 2\lVert\mathbf v\rVert^2 + 2\lVert\mathbf w\rVert^2 .$$

So L1 and L$\infty$ measure length perfectly well but carry no notion of angle: everything from
Section 5 onwards — angles, orthogonality, projection, least squares, Gram-Schmidt — is a privilege
of the inner product, not of merely having a length. Below we test all three norms against the
axioms and the parallelogram law, then draw their **unit balls**.


In [ ]:
# Three norms on one EMG feature vector (RMS amplitude per channel, arbitrary units).
emg = np.array([0.42, -1.30, 0.05, 0.88])
for p, label in [(1, "L1  (sum of |x_i|)"), (2, "L2  (Euclidean)"), (np.inf, "Linf (max |x_i|)")]:
    print(f"{label:22s}: {np.linalg.norm(emg, p):.4f}")

rng = np.random.default_rng(0)
V, W = rng.normal(size=(4000, 4)), rng.normal(size=(4000, 4))
print("\naxioms and the parallelogram law, over 4000 random pairs:")
for p, name in [(1, "L1"), (2, "L2"), (np.inf, "Linf")]:
    nv = np.linalg.norm(V, p, axis=1); nw = np.linalg.norm(W, p, axis=1)
    homog   = np.abs(np.linalg.norm(-2.5*V, p, axis=1) - 2.5*nv).max()
    triang  = (np.linalg.norm(V + W, p, axis=1) - (nv + nw)).max()
    par_lhs = np.linalg.norm(V + W, p, axis=1)**2 + np.linalg.norm(V - W, p, axis=1)**2
    par_gap = np.abs(par_lhs - 2*nv**2 - 2*nw**2).max()
    print(f"  {name:4s} homogeneity err {homog:.1e} | triangle violation {triang:+.1e}"
          f" | parallelogram gap {par_gap:8.3f}"
          + ("   <- induced by an inner product" if par_gap < 1e-9 else ""))


In [ ]:
# The unit balls: all three norms agree on the axes and disagree everywhere else.
th = np.linspace(0, 2*np.pi, 400)
fig, ax = plt.subplots(figsize=(4.6, 4.6))
ax.plot(np.cos(th), np.sin(th), color="#0A5453", lw=2.2, label="||x||_2 <= 1")
ax.plot([1, 0, -1, 0, 1], [0, 1, 0, -1, 0], color="#1B6CA8", lw=2, label="||x||_1 <= 1")
ax.plot([1, 1, -1, -1, 1], [1, -1, -1, 1, 1], color="#C77700", lw=2, ls="--", label="||x||_inf <= 1")
ax.scatter([1, 0, -1, 0], [0, 1, 0, -1], color="#1B6CA8", zorder=5)
ax.axhline(0, color="gray", lw=0.6); ax.axvline(0, color="gray", lw=0.6)
ax.set_aspect("equal"); ax.set_xlim(-1.45, 1.75); ax.set_ylim(-1.45, 1.45)
ax.legend(loc="upper right", fontsize=8)
ax.set_title("Unit ball of each norm")
plt.show()

diag = np.array([1.0, 1.0])/np.sqrt(2)      # a direction at 45 degrees
for p, name in [(1, "L1"), (2, "L2"), (np.inf, "Linf")]:
    edge = diag/np.linalg.norm(diag, p)     # where the unit ball meets that direction
    print(f"{name:4s} boundary along the diagonal: {edge.round(3)}")
print("The L1 diamond's corners sit ON the axes -- the geometric reason an L1 penalty (Lasso)")
print("drives coefficients exactly to zero, while the round L2 ball (Ridge) only shrinks them.")


**Exercise 4.** A wearable IMU on a patient's wrist records one sample of 3-axis acceleration `a = np.array([0.8, -0.3, 9.9])` (m/s^2, x/y/z). Compute its L2 norm (the total acceleration magnitude) and the unit vector giving its direction. Then compute the L1 norm and add a one-line comment on what it measures physically compared to the L2 norm.

> 🤖 *Gemini tip:* "I have a 3-D acceleration vector in NumPy — show me how to compute its magnitude (L2 norm) and a unit vector in the same direction, and explain when I'd use the L1 norm instead."

In [ ]:
a = np.array([0.8, -0.3, 9.9])

# Your code here

**Exercise 5.** A 4-channel EMG feature vector for one time window is
`f = np.array([0.42, -1.30, 0.05, 0.88])`. (a) Report its L1, L2 and L$\infty$ norms and say which
channel the L$\infty$ norm is reading. (b) Normalise `f` to unit L2 length and confirm the result has
norm 1. (c) Test the parallelogram law for L2 and for L1 on the pair `v = np.array([1.0, 0.0])`,
`w = np.array([0.0, 1.0])` and state which of the two can support a notion of angle.

> 🤖 *Gemini tip:* "Show me how to compute the L1, L2 and L-infinity norms of a NumPy vector, and write a short check of the parallelogram law ||v+w||^2 + ||v-w||^2 = 2||v||^2 + 2||w||^2 for each of them."


In [ ]:
f = np.array([0.42, -1.30, 0.05, 0.88])
v, w = np.array([1.0, 0.0]), np.array([0.0, 1.0])

# Your code here

---
## 5 · Geometric meaning: angle, orthogonality, cosine similarity

Geometrically, `x·y = |x||y|cos(theta)`, so the dot product measures **alignment**: zero means perpendicular, and the **cosine similarity** is the standard way to compare two feature vectors (or two gene-expression profiles).


In [ ]:
def cosine(a, b):
    return (a @ b) / (np.linalg.norm(a) * np.linalg.norm(b))

a = np.array([1.0, 2.0, 2.0])
b = np.array([2.0, 1.0, -2.0])
print("a . b      :", a @ b)                  # 0 -> orthogonal
print("orthogonal :", np.isclose(a @ b, 0))
# Two gene-expression-like profiles: similar vs opposite
p1 = np.array([5.0, 1.0, 0.2, 3.0])
p2 = np.array([4.5, 1.2, 0.1, 2.8])           # similar pattern
p3 = np.array([0.1, 4.0, 5.0, 0.2])           # different pattern
print("cos(p1,p2) :", round(cosine(p1, p2), 3), "(aligned)")
print("cos(p1,p3) :", round(cosine(p1, p3), 3), "(divergent)")

**Exercise 6.** Two gene-expression profiles across two marker genes are `u = [1, 0]` (the first
marker only) and `v = [1, 1]` (both markers equally). Compute the angle (in degrees) between them
from their cosine similarity — the standard way to compare two expression profiles. *Hint:*
`np.degrees(np.arccos(...))`. Then say in one line why doubling every value in `v` would leave the
answer unchanged.

> 🤖 *Gemini tip:* "Given two 2-D NumPy vectors, show me how to find the angle between them in degrees using the cosine similarity formula, and explain why cosine similarity ignores overall magnitude."


In [ ]:
u = np.array([1.0, 0.0]); v = np.array([1.0, 1.0])

# Your code here

---
## 6 · Projection

The projection of `y` onto the line through `x` is `(x·y / x·x) x`; the leftover residual is orthogonal
to `x`. This split — "along" plus "perpendicular" — is exactly what least squares does (Notebook 2).

In [ ]:
def proj(y, x):
    return (x @ y) / (x @ x) * x

y = np.array([2.0, 3.0]); x = np.array([1.0, 0.0])
p = proj(y, x); r = y - p
print("projection:", p, "| residual:", r, "| residual . x =", round(r @ x, 6))

# visualise
fig, ax = plt.subplots(figsize=(5, 5))
ax.quiver(0, 0, *x, angles="xy", scale_units="xy", scale=1, color="black", label="x")
ax.quiver(0, 0, *y, angles="xy", scale_units="xy", scale=1, color="steelblue", label="y")
ax.quiver(0, 0, *p, angles="xy", scale_units="xy", scale=1, color="crimson", label="proj_x(y)")
ax.plot([p[0], y[0]], [p[1], y[1]], "--", color="gray")
ax.set_xlim(-0.5, 3); ax.set_ylim(-0.5, 3.5); ax.set_aspect("equal"); ax.grid(alpha=0.3); ax.legend()
ax.set_title("Projection of y onto x"); plt.show()

**Exercise 7.** A physical therapist measures the raw pulling force a patient exerts, `F = np.array([4.0, 3.0])` N, while a resistance band is oriented along `d = np.array([1.0, 0.0])`. Compute the component of `F` that actually acts along the band (the projection) and the wasted perpendicular component (the residual). Report the magnitude of each.

> 🤖 *Gemini tip:* "Given a 2-D force vector and a direction vector in NumPy, show me how to split the force into the part along that direction and the part perpendicular to it, using the projection formula."

In [ ]:
F = np.array([4.0, 3.0])
d = np.array([1.0, 0.0])

# Your code here

---
## 7 · Gram-Schmidt orthonormalization

Given linearly independent vectors, Gram-Schmidt builds an **orthonormal basis** spanning the same subspace: take each vector in turn, subtract its projection onto every previous (already-orthonormal) direction — exactly the projection of Section 6, repeated — then normalise to unit length. Orthonormal bases are the backbone of QR decomposition, stable least squares, and PCA (Notebook 2); here we use one to make sense of a set of correlated biosignal channels.

In [ ]:
def gsBasis(A):
    """Classical Gram-Schmidt: build an orthonormal basis for the columns of A."""
    B = np.array(A, dtype=float)
    for i in range(B.shape[1]):
        for j in range(i):
            B[:, i] = B[:, i] - (B[:, i] @ B[:, j]) * B[:, j]   # subtract projection onto q_j
        norm = np.linalg.norm(B[:, i])
        B[:, i] = B[:, i] / norm if norm > 1e-12 else np.zeros_like(B[:, i])
    return B

# Four correlated 'lead' vectors from a toy multi-channel biosignal recording
V = np.array([
    [1.0,  0.0, 2.0, 6.0],
    [0.0,  1.0, 8.0, 2.0],
    [2.0,  8.0, 3.0, 1.0],
    [1.0, -6.0, 2.0, 3.0],
])
Q = gsBasis(V)
print("Q.T @ Q = I ?", np.allclose(Q.T @ Q, np.eye(4)))
print(Q.round(3))

A recording channel that is really a duplicate of two others (a wiring fault, or a redundant lead) shows up as a **zero column** after Gram-Schmidt — a numerical way to detect it and read off the true (lower) dimensionality of the signal.

In [ ]:
# Channel 3 here is really channel 1 - 3*channel 2 (a duplicated/redundant lead)
C = np.array([
    [1.0, 0.0,  2.0],
    [0.0, 1.0, -3.0],
    [1.0, 0.0,  2.0],
])
Qc = gsBasis(C)
print(Qc.round(3))
true_rank = int(np.sum(np.linalg.norm(Qc, axis=0) > 1e-8))
print("effective (numeric) rank of the 3 channels:", true_rank)

**Exercise 8.** `gsBasis` above orthonormalises the recording-channel directions with a nested loop
(`for i ... for j < i`). Write a **vectorized** version `gsBasisV(A)` that, for each column `i`,
subtracts the projection onto *all* previous orthonormal columns in one shot —
`B[:, :i] @ (B[:, :i].T @ B[:, i])` — instead of looping over `j`. Confirm it agrees with `gsBasis`
on `V` above, then compare the two on a random `400 x 300` matrix with `%timeit` — the size a
high-density EMG or EEG array reaches, where the difference stops being academic.

> 🤖 *Gemini tip:* "Rewrite a Gram-Schmidt loop so the projection onto all previously computed orthonormal vectors is a single matrix-vector product, and show me how to time both versions with %timeit."


In [ ]:
rng = np.random.default_rng(0)
big = rng.random((400, 300))

# Your code here

With an orthonormal basis in hand, projecting onto the subspace it spans is just a couple of dot products: express the vector in that basis, drop the coordinates you don't want, map back. This is the general recipe behind every projection/reflection/rotation in the rest of this notebook, now applied to an arbitrary (not axis-aligned) subspace.

In [ ]:
# Project a new noisy reading onto the 2-D subspace spanned by the first two lead directions
reading = np.array([3.0, -1.0, 4.0, 0.5])
Qsub = Q[:, :2]                      # orthonormal basis of the target subspace
coeffs = Qsub.T @ reading            # coordinates in that subspace (Qsub is orthonormal)
projected = Qsub @ coeffs            # back to the original 4-D space
residual = reading - projected
print("coordinates in the 2 template directions:", coeffs.round(3))
print("projected reading:", projected.round(3))
print("residual orthogonal to the subspace ?", np.allclose(Qsub.T @ residual, 0))

---
## 8 · Matrices as maps (columns = images of the basis)

Read a matrix as a **verb**. The columns of `A` are where it sends the standard basis vectors `e1, e2`,
so `A @ x` is a weighted sum of the columns. We will *see* this throughout the zoo below.

In [ ]:
A = np.array([[2.0, 1.0],
              [0.0, 3.0]])
e1, e2 = np.array([1.0, 0.0]), np.array([0.0, 1.0])
print("A @ e1 =", A @ e1, "= first column ", A[:, 0])
print("A @ e2 =", A @ e2, "= second column", A[:, 1])

**Shapes, and how the book stores your data.** An $m\times n$ matrix times an $n\times p$ matrix is
$m\times p$: the inner dimensions must match and they vanish, the outer two survive. Read `A @ x` the
same way — $(m\times n)(n\times 1) = m\times 1$ — and a shape error is telling you the two objects do
not compose, not that NumPy is being fussy.

One convention is worth fixing now because Géron and scikit-learn use it everywhere: a dataset is
stored **one sample per row**, `X.shape == (n_samples, n_features)`. So the prediction of a linear
model on the whole dataset is `X @ w`, an $(m\times n)(n\times 1)$ product giving one number per
sample — *not* `W @ x` as in the neural-network layers of Module 3, where a single sample is a column.
Both are the same mathematics; only the storage order differs. When a shape error appears, ask which
convention the object came from.


**Exercise 9.** Mechanical crosstalk between the two axes of a low-cost accelerometer can be modelled as a matrix `A = np.array([[1.10, 0.05], [0.02, 0.95]])` acting on the true signal. Find where it sends `e1` and `e2` (its columns), then apply it to a true reading `x = np.array([2.0, -1.0])` to see the (slightly corrupted) sensor output `A @ x`.

> 🤖 *Gemini tip:* "Explain what the columns of a 2x2 NumPy matrix tell me about where it sends the standard basis vectors, then show me how to apply the matrix to an arbitrary vector."

In [ ]:
Across = np.array([[1.10, 0.05], [0.02, 0.95]])
x_true = np.array([2.0, -1.0])

# Your code here

---
## 9 · The transformation zoo (visualised)

We apply each canonical 2-D map to a small asymmetric shape (so rotations and reflections are obvious)
and to the basis vectors. The shape stands in for a configuration of anatomical landmarks — exactly the
kind of object you rotate/reflect when **registering** medical images or aligning a robot frame.

In [ ]:
# An asymmetric 'flag' shape as a set of 2-D points (columns are points)
shape = np.array([
    [0, 0, 1.5, 1.5, 0.0, 0.0],   # x
    [0, 2, 2.0, 1.4, 1.4, 0.0],   # y
])

def plot_map(A, title, ax):
    new = A @ shape
    ax.plot(*np.c_[shape, shape[:, :1]], color="lightgray", lw=2, label="original")
    ax.plot(*np.c_[new,   new[:, :1]],   color="crimson",  lw=2, label="mapped")
    # basis vectors and their images
    for v, c in [(np.array([1,0]), "black"), (np.array([0,1]), "black")]:
        ax.quiver(0,0,*(A@v), angles="xy", scale_units="xy", scale=1, color="navy", width=0.012)
    ax.set_title(title, fontsize=10); ax.set_aspect("equal"); ax.grid(alpha=0.3)
    ax.set_xlim(-3, 3.5); ax.set_ylim(-3, 3.5)

theta = np.radians(40)
maps = {
    "Scaling (2, 0.5)":    np.array([[2,0],[0,0.5]]),
    "Rotation 40 deg":     np.array([[np.cos(theta),-np.sin(theta)],[np.sin(theta),np.cos(theta)]]),
    "Reflection (x-axis)": np.array([[1,0],[0,-1]]),
    "Shear (k=1)":         np.array([[1,1],[0,1]]),
    "Projection (x-axis)": np.array([[1,0],[0,0]]),
    "Identity":            np.eye(2),
}
fig, axes = plt.subplots(2, 3, figsize=(12, 8))
for ax, (name, M) in zip(axes.ravel(), maps.items()):
    plot_map(M, name, ax)
axes[0,0].legend(fontsize=8)
plt.tight_layout(); plt.show()

**Exercise 10.** A microscope slide has been loaded into the stage a quarter-turn out of true, so
every landmark in the captured frame must be rotated by 90 degrees before it can be compared with
the reference atlas. Build the matrix that rotates by 90 degrees, apply it to `shape`, and confirm
a landmark that was at `(1.5, 0)` lands at `(0, 1.5)`.

> 🤖 *Gemini tip:* "Show me how to build a 2-D rotation matrix for a specific angle in NumPy and apply it to a set of points stored as columns."


In [ ]:
# Your code here

---
## 10 · Composition and non-commutativity

Multiplying matrices **composes** maps, applied right-to-left, and order matters: rotate-then-reflect is
not reflect-then-rotate.

In [ ]:
R = np.array([[np.cos(theta), -np.sin(theta)], [np.sin(theta), np.cos(theta)]])
F = np.array([[1.0, 0.0], [0.0, -1.0]])     # reflect in x-axis
print("R @ F =\n", R @ F)
print("F @ R =\n", F @ R)
print("equal?", np.allclose(R @ F, F @ R))  # False -> order matters

**Exercise 11.** Registering two medical images sometimes requires a rotation *and* a reflection (e.g. one scan was mirrored during acquisition). Build a 30-degree rotation matrix `R30` and reuse the horizontal-reflection matrix `F` from above, then apply *rotate-then-reflect* and *reflect-then-rotate* to the anatomical landmark point `p = np.array([2.0, 1.0])`. Confirm the two pipelines give different results.

> 🤖 *Gemini tip:* "I have two 2x2 NumPy transformation matrices; show me how to apply them to a point in two different orders and confirm matrix multiplication doesn't commute."

In [ ]:
p = np.array([2.0, 1.0])

# Your code here

---
## 11 · Orthogonal matrices and rigid motions

An **orthogonal** matrix satisfies `Q.T @ Q = I` (so its inverse is its transpose). It preserves lengths
and angles — a rigid motion. `det(Q) = +1` is a **rotation**, `det(Q) = -1` a **reflection**. This is the
$SO(3)$ structure used to orient a robot or a surgical tool.

In [ ]:
Q = R                          # a rotation
print("Q.T @ Q = I ?", np.allclose(Q.T @ Q, np.eye(2)))
print("Q^-1 == Q.T ?", np.allclose(np.linalg.inv(Q), Q.T))
print("det(rotation)  =", round(np.linalg.det(Q), 3))    # +1
print("det(reflection)=", round(np.linalg.det(F), 3))    # -1
# length preservation
w = np.array([3.0, 4.0])
print("|w| =", np.linalg.norm(w), "| |Qw| =", round(np.linalg.norm(Q @ w), 3))

**Exercise 12.** A colleague claims a candidate calibration matrix `C = np.array([[0.8, 0.6], [0.6, -0.8]])` represents a valid rigid rotation of a surgical tool's reference frame. Check whether `C` is orthogonal (`C.T @ C == I`) and compute `det(C)`. Is it a rotation, a reflection, or neither?

> 🤖 *Gemini tip:* "Given a 2x2 NumPy matrix, show me how to test whether it's orthogonal and how the sign of its determinant tells me if it's a rotation or a reflection."

In [ ]:
C = np.array([[0.8, 0.6], [0.6, -0.8]])

# Your code here

---
## 12 · The determinant as area scaling, and rank

`det(A)` is the factor by which the map scales area (with sign = orientation). A zero determinant means
the map collapses the plane onto a line — it is singular and cannot be inverted.

The **rank** is the dimension of the output space — how many independent directions survive the map.
A full-rank map is invertible; a rank-deficient one (like a projection) has collapsed at least one
direction, destroying information that cannot be recovered — in data terms, rank-deficient columns
(features) are redundant, which is exactly what dimensionality reduction exploits.

In [ ]:
for name, M in [("scaling(2,0.5)", np.array([[2,0],[0,0.5]])),
                ("rotation",       R),
                ("shear(k=1)",     np.array([[1,1],[0,1]])),
                ("projection",     np.array([[1,0],[0,0]]))]:
    print(f"{name:16s} det = {np.linalg.det(M): .3f}  | rank = {np.linalg.matrix_rank(M)}")
print("\nUnit square has area 1; after scaling(2,0.5) it has area", abs(np.linalg.det(np.array([[2,0],[0,0.5]]))))
print("Note: projection has det = 0 and rank 1 -- it collapses the plane onto a line, so it cannot be inverted.")

**Exercise 13.** A microscope's pixel grid is not perfectly square: converting pixel coordinates to physical (µm) coordinates uses the anisotropic scaling matrix `A = np.array([[1.2, 0.0], [0.0, 0.8]])`. Compute `det(A)` and use it to convert a measured cell area of `150` pixels^2 into µm^2.

> 🤖 *Gemini tip:* "Explain how the determinant of a 2x2 scaling matrix relates to how areas change under that transformation, and show me how to use it to convert a pixel area into physical units."

In [ ]:
Apix = np.array([[1.2, 0.0], [0.0, 0.8]])
area_px = 150

# Your code here

**Exercise 14 — Rank.** A resting-state fMRI connectivity map that has collapsed onto a single dominant
mode can be modelled by the rank-deficient matrix `Mp = np.array([[1, 0], [0, 0]])` (all signal projected
onto one axis). Compute `np.linalg.matrix_rank(Mp)` and `np.linalg.det(Mp)`, and in one comment explain what
a rank-1, determinant-0 matrix means for trying to invert it — i.e. recover the discarded second axis from
the output alone.

> 🤖 *Gemini tip:* "Given a 2x2 NumPy matrix, show me how to compute its rank and determinant, and explain in plain terms what it means when a matrix is rank-deficient with determinant zero."


In [ ]:
Mp = np.array([[1, 0], [0, 0]])   # rank-1 projection onto the x-axis

# Your code here

---
## 13 · Change of basis: the $\boldsymbol\Gamma$ notation

A point's coordinates depend entirely on the basis you measure them in — the same arrow is a
different list of numbers in a different basis. Let $B$ and $E$ be two bases of the same space. Write
each vector of $E$ in $B$-coordinates and stack those columns into the **basis-change matrix**

$$\boldsymbol\Gamma_{BE} \;=\; \big[\;[\mathbf e_1]_B \;\; [\mathbf e_2]_B \;\cdots\; [\mathbf e_n]_B\;\big],
\qquad\text{so that}\qquad
\mathbf v_B \;=\; \boldsymbol\Gamma_{BE}\,\mathbf v_E .$$

The naming is mnemonic: read the subscript left to right and $\boldsymbol\Gamma_{BE}$ *expresses the
vectors of the second letter as coordinates in the basis of the first*. The matrix runs "the other
way" on purpose — it **takes in** $E$-coordinates and **returns** $B$-coordinates — because its
columns were built *from* the $E$-vectors expressed *in* $B$. Basis-change matrices are square and
invertible, so $\boldsymbol\Gamma_{EB} = \boldsymbol\Gamma_{BE}^{-1}$, and when $E$ is the canonical
basis $\boldsymbol\Gamma_{EB}$ is simply the new basis vectors written as columns. A map changes with
a **similarity transform**, $\mathbf A_B = \boldsymbol\Gamma_{BE}\,\mathbf A_E\,\boldsymbol\Gamma_{EB}$
— the idea PCA uses to find the axes in which data is simplest.


In [ ]:
# E is the canonical basis; B is a skewed basis of sensor directions.
b1, b2 = np.array([1.0, 1.0]), np.array([1.0, -1.0])
Gam_EB = np.column_stack([b1, b2])          # B's vectors written in E-coordinates
Gam_BE = np.linalg.inv(Gam_EB)              # the inverse translates the other way
print("Gamma_EB (B's vectors as columns):\n", Gam_EB)
print("Gamma_BE = inv(Gamma_EB):\n", Gam_BE)

v_E = np.array([3.0, 1.0])                  # one arrow, measured in E
v_B = Gam_BE @ v_E                          # the same arrow, measured in B
print("\nv in E-coordinates:", v_E)
print("v in B-coordinates:", v_B, " -> v =", v_B[0], "* b1 +", v_B[1], "* b2")
print("round trip Gamma_EB @ v_B:", Gam_EB @ v_B, " (recovers v_E)")
print("Gamma_BE @ Gamma_EB = I:", np.allclose(Gam_BE @ Gam_EB, np.eye(2)))


In [ ]:
# One arrow, two grids. Nothing about the vector changes -- only the ruler.
fig, (axE, axB) = plt.subplots(1, 2, figsize=(10, 4.4))

for ax, title in [(axE, "measured against E (canonical)"), (axB, "measured against B (skewed)")]:
    ax.axhline(0, color="gray", lw=0.6); ax.axvline(0, color="gray", lw=0.6)
    ax.set_xlim(-0.6, 4.2); ax.set_ylim(-1.6, 3.2); ax.set_aspect("equal")
    ax.set_title(title, fontsize=10); ax.grid(False)

for i in range(-2, 6):                                   # the two grids
    axE.plot([i, i], [-2, 4], color="#BBBBBB", lw=0.6)
    axE.plot([-2, 6], [i, i], color="#BBBBBB", lw=0.6)
for i in range(-3, 6):
    p0, p1 = i*b1 - 3*b2, i*b1 + 3*b2
    axB.plot([p0[0], p1[0]], [p0[1], p1[1]], color="#BBBBBB", lw=0.6)
    p0, p1 = i*b2 - 3*b1, i*b2 + 3*b1
    axB.plot([p0[0], p1[0]], [p0[1], p1[1]], color="#BBBBBB", lw=0.6)

for ax in (axE, axB):
    ax.annotate("", xy=v_E, xytext=(0, 0),
                arrowprops=dict(arrowstyle="-|>", lw=2.4, color="#0A5453"))
    ax.text(v_E[0]+0.08, v_E[1]+0.12, "v", color="#0A5453", fontsize=13)
axE.annotate("", xy=(3, 0), xytext=(0, 0), arrowprops=dict(arrowstyle="-|>", color="#C77700", lw=1.8))
axE.annotate("", xy=v_E, xytext=(3, 0), arrowprops=dict(arrowstyle="-|>", color="#C77700", lw=1.8))
axE.text(1.4, -0.45, "3 e1", color="#C77700"); axE.text(3.1, 0.45, "1 e2", color="#C77700")
axB.annotate("", xy=2*b1, xytext=(0, 0), arrowprops=dict(arrowstyle="-|>", color="#1B6CA8", lw=1.8))
axB.annotate("", xy=v_E, xytext=2*b1, arrowprops=dict(arrowstyle="-|>", color="#1B6CA8", lw=1.8))
axB.text(0.8, 2.2, "2 b1", color="#1B6CA8"); axB.text(2.8, 1.6, "1 b2", color="#1B6CA8")
axE.set_xlabel("v_E = (3, 1)"); axB.set_xlabel("v_B = (2, 1)")
plt.tight_layout(); plt.show()


### 13.1 · A basis change is not a linear map (even when the numbers coincide)

Both objects are square matrices and both get written $\mathbf y = \mathbf M\mathbf x$, which is why
they are so easily confused. The difference is what changes:

* a **basis change** leaves the vector alone and changes the *description* — one arrow, two coordinate lists;
* a **linear map** leaves the basis alone and moves the *vector* — two arrows, one coordinate system.

The recipes even look alike. $\boldsymbol\Gamma_{EB}$ collects the new basis vectors in
$E$-coordinates as columns; the matrix $\mathbf A_E$ of a map $T$ collects the *images* $T(\mathbf
e_j)$ in $E$-coordinates as columns. So a single array of numbers can play either role, and only the
context tells you which. Below, the very same matrix $[\,\mathbf b_1\ \ \mathbf b_2\,]$ is used both
ways, and the two results are genuinely different things.


In [ ]:
M = Gam_EB.copy()      # the same array of numbers, used twice

# reading 1: M is Gamma_EB, a basis change. The arrow is fixed; we re-describe it.
u_B = np.array([2.0, 1.0])                 # a vector's coordinates IN B
u_E = M @ u_B                              # the SAME vector's coordinates in E
print("basis change : u_B =", u_B, "-> u_E =", u_E, " (one arrow, two descriptions)")

# reading 2: M is A_E, the matrix of a linear map T in the canonical basis. The arrow moves.
x_E = np.array([2.0, 1.0])                 # a vector in E
y_E = M @ x_E                              # a DIFFERENT vector, the image T(x)
print("linear map   : x_E =", x_E, "-> T(x)_E =", y_E, " (two arrows, one description)")
print("the numbers agree, the meaning does not:",
      "u_E and T(x)_E are", np.allclose(u_E, y_E), "identical as arrays")

# and a map changes description by a similarity transform
A_E = np.array([[2.0, 0.0], [0.0, 0.5]])   # stretch x, squash y -- in canonical coordinates
A_B = Gam_BE @ A_E @ Gam_EB                # the same map, described in B
print("\nA_E =\n", A_E, "\nA_B = Gamma_BE A_E Gamma_EB =\n", A_B.round(4))
z_E = np.array([1.0, 3.0])
print("map then convert:", (Gam_BE @ (A_E @ z_E)).round(4))
print("convert then map:", (A_B @ (Gam_BE @ z_E)).round(4), " -> the same, as they must be")


**Exercise 15.** Two skin electrodes define a non-orthogonal measurement frame
`b1 = np.array([2.0, 1.0])`, `b2 = np.array([0.0, 3.0])` (in canonical mm coordinates).
(a) Build `Gamma_EB` and `Gamma_BE` and verify they are inverses. (b) A dipole source sits at
`v_E = np.array([4.0, 6.0])`; find `v_B` and check by expanding `v_B[0]*b1 + v_B[1]*b2`.
(c) The gain of the amplifier chain is the map `A_E = np.diag([1.0, 0.5])` in canonical coordinates;
write it in the electrode frame with a similarity transform, and confirm that converting-then-mapping
and mapping-then-converting give the same answer for `v_E`.

> 🤖 *Gemini tip:* "Explain the difference between a change-of-basis matrix and the matrix of a linear map, and show me NumPy code that converts a vector's coordinates between two bases and rewrites a linear map with a similarity transform."


In [ ]:
b1e = np.array([2.0, 1.0]); b2e = np.array([0.0, 3.0])
vE  = np.array([4.0, 6.0])
AE  = np.diag([1.0, 0.5])

# Your code here

### 13.2 · Choosing an easy basis: the conjugation trick

The similarity transform above, $\mathbf A_E=\boldsymbol\Gamma_{EB}\,\mathbf A_B\,\boldsymbol\Gamma_{BE}$,
is usually met as bad news: a map's matrix depends on the ruler you hold up to it. Read the other way
round it is the most useful technique in this notebook — **you get to choose the ruler.**

Plenty of geometric operations are awkward to write in canonical coordinates: a mirror across a tilted
line, a rotation about an oblique axis, a projection onto a plane that is not a coordinate plane. So do
not write them there. Instead:

1. **Choose a basis adapted to the map** — vectors along the directions it treats specially (the mirror
   line, the rotation axis, the plane you project onto) plus their perpendiculars. Make it
   **orthonormal**, so that $\boldsymbol\Gamma_{EB}=\mathbf Q$ and $\boldsymbol\Gamma_{BE}=\mathbf Q^{\mathsf T}$
   — transposing is free, inverting is not.
2. **Write $\mathbf A_B$ by inspection.** In that basis it is diagonal, or a single $2\times2$ rotation block.
3. **Assemble** $\;\mathbf A_E=\mathbf Q\,\mathbf A_B\,\mathbf Q^{\mathsf T}$.

Read right to left, the formula is a three-step pipeline:

$$\mathbf x_E \;\xrightarrow{\;\;\mathbf Q^{\mathsf T}\;\;}\; \mathbf x_B
\;\xrightarrow{\;\;\mathbf A_B\ \text{(easy)}\;\;}\; (\mathbf A\mathbf x)_B
\;\xrightarrow{\;\;\mathbf Q\;\;}\; (\mathbf A\mathbf x)_E$$

The two outer arrows are pure relabelling — **nothing moves**. All of the geometry happens in the middle.
This is §8.2 of the Module 2 PDF; the three examples below are its three worked cases, set in
biomechanics, and each is followed by an exercise.

In [ ]:
def assemble(Q, A_B):
    """The conjugation trick: the map that is A_B in the orthonormal basis Q, written in canonical coordinates."""
    return Q @ A_B @ Q.T


def frame_around(a):
    """An orthonormal, right-handed basis [a, u, v] of R^3 whose FIRST vector points along a."""
    a = np.asarray(a, dtype=float)
    a = a / np.linalg.norm(a)
    t = np.array([1.0, 0.0, 0.0]) if abs(a[0]) < 0.9 else np.array([0.0, 1.0, 0.0])
    u = t - (t @ a) * a            # Gram-Schmidt: remove the part of t along a
    u = u / np.linalg.norm(u)
    v = np.cross(a, u)             # completes a right-handed frame
    return np.column_stack([a, u, v])


Q_test = frame_around([1, 2, 2])
print("frame is orthonormal:", np.allclose(Q_test.T @ Q_test, np.eye(3)),
      "| right-handed (det = +1):", np.isclose(np.linalg.det(Q_test), 1.0))
print("first column points along a:", np.round(Q_test[:, 0], 4), "vs", np.round(np.array([1, 2, 2]) / 3, 4))

**Example 1 — Mirroring a foot scan (reflection).** A pressure-mat scan of a patient's *right* foot is
taken with the body midline tilted, running at $\alpha=70^\circ$ to the mat's $x$-axis. The orthotist needs
the matching *left* insole: the mirror image of the outline across that midline.

In canonical coordinates the reflection is a tangle of $\sin$ and $\cos$. In the basis
$\{\mathbf b_1 \text{ along the midline},\ \mathbf b_2 \text{ across it}\}$ it is simply
$\operatorname{diag}(1,-1)$ — keep the along-midline coordinate, flip the across-midline one.

In [ ]:
# ---- a chiral foot outline, in the foot's own coordinates (cm): long axis along +y, heel at y = 0
t = np.linspace(0, 2 * np.pi, 400)
L = 25.0
y_loc = L / 2 * (1 - np.cos(t))                                   # heel (0) -> toe (L) -> heel
half_w = 3.0 + 1.9 * np.sin(np.pi * y_loc / L) ** 1.5              # narrow heel, broad forefoot
bulge = np.where(np.sin(t) > 0, 1.0, 0.0) * np.exp(-((y_loc - 19.0) / 3.5) ** 2)
x_loc = np.sin(t) * half_w * (1 + 0.38 * bulge)                    # ball-of-foot bulge on ONE side -> chiral
foot_local = np.vstack([x_loc + 9.0, y_loc - 2.0])                 # sits 9 cm to one side of the midline

# ---- place it on the mat: the midline runs at alpha to the mat's x-axis
alpha = np.radians(70)
b1 = np.array([np.cos(alpha), np.sin(alpha)])                     # along the midline
b2 = np.array([-np.sin(alpha), np.cos(alpha)])                    # across the midline
theta = alpha - np.pi / 2                                          # rotates the foot's +y onto b1
R_place = np.array([[np.cos(theta), -np.sin(theta)], [np.sin(theta), np.cos(theta)]])
right_foot = R_place @ foot_local

# ---- the conjugation trick
Q = np.column_stack([b1, b2])          # adapted, orthonormal basis
A_B = np.diag([1.0, -1.0])             # a mirror, written where it is obvious
M = assemble(Q, A_B)                   # ... and assembled in mat coordinates
left_foot = M @ right_foot

print("mirror matrix in mat coordinates:\n", M.round(4))
closed = np.array([[np.cos(2 * alpha), np.sin(2 * alpha)], [np.sin(2 * alpha), -np.cos(2 * alpha)]])
print("equals the closed form [[cos2a, sin2a],[sin2a, -cos2a]]:", np.allclose(M, closed))
print(f"det = {np.linalg.det(M):+.3f}  (a reflection)  |  M@M = I: {np.allclose(M @ M, np.eye(2))}"
      f"  |  midline fixed: {np.allclose(M @ b1, b1)}  |  normal flipped: {np.allclose(M @ b2, -b2)}")

fig, ax = plt.subplots(figsize=(6.8, 5.8))
s = np.linspace(-9, 32, 2)
ax.plot(s * b1[0], s * b1[1], "k--", lw=1, label="body midline (mirror line)")
ax.fill(*right_foot, color="steelblue", alpha=0.45, label="scanned: right foot")
ax.fill(*left_foot, color="darkorange", alpha=0.45, label="mirrored: left insole")
ax.plot(*right_foot[:, :1], "o", color="steelblue")
ax.plot(*left_foot[:, :1], "o", color="darkorange")
ax.annotate("heel", right_foot[:, 0], xytext=(6, -10), textcoords="offset points", fontsize=8)
ax.annotate("heel", left_foot[:, 0], xytext=(6, -10), textcoords="offset points", fontsize=8)
ax.set_aspect("equal"); ax.set_xlim(-22, 26); ax.set_ylim(-9, 31)
ax.set_xlabel("mat x (cm)"); ax.set_ylabel("mat y (cm)")
ax.set_title("Reflection across a tilted line, built as Q diag(1,-1) Q^T", fontsize=10)
ax.legend(loc="lower left", fontsize=8); ax.grid(alpha=0.3)
plt.show()

**Exercise 16.** A second patient is scanned with the midline at `alpha2 = np.radians(125)`.
(a) Build the adapted basis and the mirror matrix `M2` **by conjugation** — do not type in the closed form.
(b) Verify the three properties every reflection must have: `det(M2) = -1`, `M2 @ M2 = I` (mirroring
twice returns the original foot), and that the midline direction is left unchanged.
(c) Only then compare `M2` against the closed form $\begin{bmatrix}\cos2\alpha&\sin2\alpha\\ \sin2\alpha&-\cos2\alpha\end{bmatrix}$.

> 🤖 *Gemini tip:* "Show me how to build a 2D reflection matrix across a line at angle alpha in NumPy as Q @ diag(1,-1) @ Q.T, and how to check numerically that the result really is a reflection."

In [ ]:
alpha2 = np.radians(125)

# Your code here

**Example 2 — Knee flexion about an oblique axis (rotation).** The knee does not flex about a lab axis.
Its functional flexion–extension axis runs roughly mediolateral but is tilted by the shape of the femoral
condyles — `a` below, in lab coordinates ($x$ mediolateral, $y$ anterior, $z$ vertical) with the **knee
joint centre at the origin**, so the rotation is linear rather than affine.

Rotating the shank markers about `a` directly needs Rodrigues' formula. With the trick it needs only the
familiar $2\times2$ rotation, placed in the corner of a $3\times3$ matrix in the frame $\{\mathbf a,\mathbf u,\mathbf v\}$
where `a` is the first axis — exactly the $SO(3)$ construction of the Robotics & Bionics unit.

In [ ]:
def rot_about_first_axis(theta):
    """Rotation by theta that fixes the FIRST basis vector: a 2x2 block in the corner."""
    c, s = np.cos(theta), np.sin(theta)
    return np.array([[1, 0, 0], [0, c, -s], [0, s, c]])


a = np.array([-1.0, 0.15, 0.10]); a /= np.linalg.norm(a)     # oblique knee flexion axis (medial-pointing)
flexion = np.radians(45)

Q = frame_around(a)
R_knee = assemble(Q, rot_about_first_axis(flexion))           # built where it is easy, read in lab coords

# markers relative to the knee joint centre (cm)
hip = np.array([0.0, 0.0, 42.0])
shank = np.array([[0.0, 3.0, -5.0],      # tibial tuberosity
                  [0.0, 2.0, -20.0],     # mid-shin
                  [0.0, 0.0, -40.0]]).T  # ankle
shank_flexed = R_knee @ shank

# check against the textbook formula, and against what a rotation must do
K = np.array([[0, -a[2], a[1]], [a[2], 0, -a[0]], [-a[1], a[0], 0]])
rodrigues = np.cos(flexion) * np.eye(3) + np.sin(flexion) * K + (1 - np.cos(flexion)) * np.outer(a, a)
print("R_knee =\n", R_knee.round(4))
print("matches Rodrigues' formula:", np.allclose(R_knee, rodrigues))
print(f"orthogonal: {np.allclose(R_knee.T @ R_knee, np.eye(3))} | det = {np.linalg.det(R_knee):+.3f}"
      f" | axis unmoved: {np.allclose(R_knee @ a, a)}")
print("shank length before / after (cm):",
      round(np.linalg.norm(shank[:, 2]), 3), "/", round(np.linalg.norm(shank_flexed[:, 2]), 3))
print("ankle moves from", shank[:, 2], "to", shank_flexed[:, 2].round(2), "-> posterior and up: flexion")

# Two views of the same 3-D scene, both at true scale: a single oblique view would either hide
# the axis end-on or distort the segment lengths, which would make the rotation look like a stretch.
fig = plt.figure(figsize=(11.5, 5.6))
views = [(4, -4, "side view: the flexion"), (22, -58, "oblique view: the tilted axis")]
for k, (elev, azim, title) in enumerate(views):
    ax = fig.add_subplot(1, 2, k + 1, projection="3d")
    ax.plot(*np.column_stack([hip, np.zeros(3)]), color="gray", lw=5, label="thigh (fixed)")
    for pts, col, lab in [(shank, "silver", "shank, extended"), (shank_flexed, "teal", "shank, flexed 45°")]:
        ax.plot(*np.column_stack([np.zeros(3), pts]), color=col, lw=4, label=lab)
        ax.scatter(*pts, color=col, s=25)
    ax.plot(*np.column_stack([-20 * a, 20 * a]), "r--", lw=1.8, label="oblique flexion axis a")
    ax.scatter([0], [0], [0], color="k", s=30)
    ax.set_xlim(-22, 22); ax.set_ylim(-40, 15); ax.set_zlim(-45, 45)
    ax.set_box_aspect((44, 55, 90))                 # equal scale on all three axes
    ax.view_init(elev=elev, azim=azim)
    ax.set_title(title, fontsize=10)
    ax.set_ylabel("y  anterior"); ax.set_zlabel("z  vertical")
    if k == 0:
        ax.set_xticks([]); ax.legend(loc="upper left", fontsize=8)
    else:
        ax.set_xlabel("x  mediolateral"); ax.set_xticks([-20, 0, 20])
fig.suptitle("Knee flexion about an oblique axis, built as Q · R(θ) · Q^T", fontsize=11)
plt.tight_layout()
plt.show()

**Exercise 17.** The Module 2 PDF (§8.2, Example 2) claims that a rotation of $120^\circ$ about the cube diagonal
$\mathbf a=\tfrac{1}{\sqrt3}(1,1,1)^{\mathsf T}$ is *exactly* the matrix that cycles
$\mathbf e_1\to\mathbf e_2\to\mathbf e_3\to\mathbf e_1$. Test the claim.
(a) Build the rotation with `frame_around` and `assemble`.
(b) Confirm it equals `[[0,0,1],[1,0,0],[0,1,0]]` to machine precision.
(c) Confirm `det = +1`, that it leaves `a` unchanged, and that applying it three times returns the
identity. In one comment, explain why *three* — and why an integer matrix appearing out of irrational
ingredients is reassuring rather than suspicious.

> 🤖 *Gemini tip:* "Show me how to build a 3D rotation matrix about an arbitrary unit axis in NumPy by building an orthonormal frame around the axis and conjugating a rotation block, then how to check it is a rotation."

In [ ]:
a_diag = np.array([1.0, 1.0, 1.0])

# Your code here

**Example 3 — Projecting gait onto the sagittal plane (projection).** Clinical gait analysis reports
motion in the **sagittal plane** — the vertical plane that contains the walking direction. But the
walkway rarely lines up with the motion-capture lab axes; here the patient walks at $25^\circ$ to the lab's
$x$-axis, so the sagittal plane is oblique in lab coordinates.

In the basis $\{\mathbf d \text{ walking direction},\ \mathbf z \text{ vertical},\ \mathbf n \text{ mediolateral}\}$
the projection is just $\operatorname{diag}(1,1,0)$: keep progression and height, discard the sideways sway.

In [ ]:
gamma = np.radians(25)
d = np.array([np.cos(gamma), np.sin(gamma), 0.0])      # walking direction (horizontal)
z = np.array([0.0, 0.0, 1.0])                         # vertical
n = np.cross(z, d)                                     # mediolateral: normal to the sagittal plane
Q_gait = np.column_stack([d, z, n])

P_sag = assemble(Q_gait, np.diag([1.0, 1.0, 0.0]))    # keep d and z, kill n

# a heel-marker trajectory over two strides (m): progression + vertical lift + mediolateral sway
s = np.linspace(0, 2.6, 300)
heel = (np.outer(d, s)
        + np.outer(z, 0.03 + 0.09 * np.clip(np.sin(2 * np.pi * s / 1.3), 0, None) ** 2)
        + np.outer(n, 0.04 * np.sin(np.pi * s / 1.3)))
heel_sag = P_sag @ heel

print("P_sag =\n", P_sag.round(4))
print(f"symmetric: {np.allclose(P_sag, P_sag.T)} | idempotent (P@P = P): {np.allclose(P_sag @ P_sag, P_sag)}"
      f" | trace = {np.trace(P_sag):.1f} (dimensions kept)")
print("max mediolateral component after projection:", float(np.abs(n @ heel_sag).max()))
print("sway removed (m): before", round(float(np.abs(n @ heel).max()), 4),
      "-> after", round(float(np.abs(n @ heel_sag).max()), 12))

# At true scale 4 cm of sway is invisible against a 2.6 m walk, so a 3-D plot would show two
# overlapping curves. Instead, read the trajectory in the adapted coordinates themselves.
along = d @ heel
fig, ax = plt.subplots(1, 3, figsize=(14.5, 4.0), gridspec_kw={"width_ratios": [1.05, 1, 1]})

ax[0].plot(heel[0], heel[1], color="darkorange", lw=1.8, label="heel marker")
ax[0].plot([0, 2.7 * d[0]], [0, 2.7 * d[1]], "k--", lw=1, label="walking direction d")
for vec, col, name, off in [(d, "teal", "d", (0.05, -0.13)), (n, "purple", "n", (-0.13, 0.03))]:
    ax[0].annotate("", xy=0.5 * vec[:2], xytext=(0, 0), arrowprops=dict(arrowstyle="->", color=col, lw=2))
    ax[0].text(0.55 * vec[0] + off[0], 0.55 * vec[1] + off[1], name, color=col, fontsize=11)
ax[0].set_aspect("equal"); ax[0].set_xlim(-0.55, 2.65); ax[0].set_ylim(-0.35, 1.45)
ax[0].set_xlabel("lab x (m)"); ax[0].set_ylabel("lab y (m)")
ax[0].set_title("top view: walkway at 25° to the lab axes", fontsize=10)
ax[0].legend(fontsize=8, loc="upper left"); ax[0].grid(alpha=0.3)

ax[1].plot(along, 100 * (n @ heel), color="darkorange", lw=2, label="before projection")
ax[1].plot(along, 100 * (n @ heel_sag), color="teal", lw=2, label="after: exactly zero")
ax[1].set_xlabel("progression along d (m)"); ax[1].set_ylabel("mediolateral (cm)")
ax[1].set_title("what the projection removes: sideways sway", fontsize=10)
ax[1].legend(fontsize=8); ax[1].grid(alpha=0.3)

ax[2].plot(along, 100 * (z @ heel_sag), color="teal", lw=2)
ax[2].set_xlabel("progression along d (m)"); ax[2].set_ylabel("heel height (cm)")
ax[2].set_title("what it keeps: the sagittal gait curve", fontsize=10); ax[2].grid(alpha=0.3)
plt.tight_layout()
plt.show()

Look again at what `assemble(Q_gait, np.diag([1, 1, 0]))` computes. Writing $\mathbf Q_k=[\mathbf d\ \ \mathbf z]$
for just the columns that survive, it equals $\mathbf Q_k\mathbf Q_k^{\mathsf T}$ — which is the least-squares
**hat matrix** $\mathbf X(\mathbf X^{\mathsf T}\mathbf X)^{-1}\mathbf X^{\mathsf T}$ of Annex A in the case where the
columns of $\mathbf X$ are already orthonormal, because then $\mathbf X^{\mathsf T}\mathbf X=\mathbf I$. Regression's
projector is not a separate invention: it is this trick applied to $\operatorname{col}(\mathbf X)$, with
$(\mathbf X^{\mathsf T}\mathbf X)^{-1}$ doing the extra work of straightening out a basis handed to you crooked.

In [ ]:
Q_k = Q_gait[:, :2]
print("P_sag equals Q_k Q_k^T:", np.allclose(P_sag, Q_k @ Q_k.T))
X_crooked = Q_k @ np.array([[2.0, 0.7], [0.0, 1.5]])        # same plane, non-orthonormal columns
hat = X_crooked @ np.linalg.inv(X_crooked.T @ X_crooked) @ X_crooked.T
print("... and equals the hat matrix X(X^T X)^-1 X^T for a crooked basis of the same plane:",
      np.allclose(P_sag, hat))

**Exercise 18.** The same lab also reports motion in the **frontal plane**, spanned by the mediolateral axis
$\mathbf n$ and the vertical $\mathbf z$.
(a) Build `P_front` by conjugation, reusing `Q_gait`.
(b) Verify it is symmetric, idempotent and has trace 2.
(c) Build `P_vert`, the projection onto the vertical line alone, the same way. Show that
`P_sag @ P_front` equals `P_vert`, and that the two plane projectors **commute**.
In one comment, explain why both facts are immediate once you look at the three matrices in the
basis `Q_gait` rather than in lab coordinates.

> 🤖 *Gemini tip:* "Two projection matrices are both diagonal in the same orthonormal basis. Explain why they commute and why their product is the projection onto the intersection of the two subspaces."

In [ ]:
# Your code here

**Why this keeps coming back.** Notebook 2 is this same move four times over, differing only in how the
easy basis is *found* rather than chosen: the **eigendecomposition** $\mathbf A=\mathbf P\boldsymbol\Lambda\mathbf P^{-1}$
(the eigenbasis, where the map is pure scaling), the **SVD** $\mathbf A=\mathbf U\boldsymbol\Sigma\mathbf V^{\mathsf T}$
(two bases, because a non-square map has no single one to conjugate with), **PCA** (the basis in which the
covariance is diagonal) and **least squares** (an orthonormal basis of $\operatorname{col}(\mathbf X)$). Four
formulas that look unrelated on a formula sheet are one idea: *find the basis in which the thing is
diagonal, work there, come back.*

---
## 14 · Vector spaces beyond $\mathbb R^n$: polynomials and matrices

Nothing in Sections 1–12 used the fact that a vector is an arrow. A **vector space** is any set with
an addition and a scalar multiplication obeying the axioms, and the coordinate machinery —
bases, coordinates, $\boldsymbol\Gamma_{BE}$ — is pure bookkeeping that never asks what the objects
are. Two spaces you already use daily make the point.

**Polynomials.** $P_2 = \{a_0 + a_1x + a_2x^2\}$ is a 3-dimensional vector space. Take
$E = \{1,\,x,\,x^2\}$ (so a polynomial's $E$-coordinates are just its coefficients) and
$B = \{1,\,1{+}x,\,1{+}x{+}x^2\}$. In bioengineering these are the calibration curves of a sensor:
same curve, two ways of writing it.

**Matrices.** $\mathbb R^{2\times2}$ is a 4-dimensional vector space. The canonical-style basis is the
four unit matrices; a more meaningful one splits every matrix into its **symmetric** and
**antisymmetric** parts, since $\mathbf M = \tfrac12(\mathbf M + \mathbf M^{\mathsf T}) + \tfrac12(\mathbf M - \mathbf M^{\mathsf T})$.
For the 2-D *displacement-gradient* tensor of deforming tissue, the symmetric part **is** the strain
and the antisymmetric part **is** the rigid rotation — so this change of basis performs a physically
meaningful decomposition, and the coordinates read out the two effects separately.


In [ ]:
# P2 in two bases. E = {1, x, x^2}; B = {1, 1+x, 1+x+x^2}.
# Columns of Gamma_EB are B's vectors written in E-coordinates (i.e. as coefficient triples).
Gp_EB = np.array([[1.0, 1.0, 1.0],
                  [0.0, 1.0, 1.0],
                  [0.0, 0.0, 1.0]])
Gp_BE = np.linalg.inv(Gp_EB)
print("Gamma_EB:\n", Gp_EB, "\nGamma_BE:\n", Gp_BE)
print("inverses:", np.allclose(Gp_BE @ Gp_EB, np.eye(3)))

p_E = np.array([1.0, 0.0, 1.0])          # p(x) = 1 + x^2, read straight off the coefficients
p_B = Gp_BE @ p_E
print("\np_E =", p_E, "  ->  p_B =", p_B)
print("check: p = %g*1 + %g*(1+x) + %g*(1+x+x^2)" % tuple(p_B))
print("       constants %g, x terms %g, x^2 term %g"
      % (p_B[0] + p_B[1] + p_B[2], p_B[1] + p_B[2], p_B[2]))


In [ ]:
# One curve, two decompositions -- both sum, point by point, to the same polynomial.
xs = np.linspace(-1.5, 1.5, 200)
basis_E = [("1", np.ones_like(xs)), ("x", xs), ("x^2", xs**2)]
basis_B = [("1", np.ones_like(xs)), ("1+x", 1 + xs), ("1+x+x^2", 1 + xs + xs**2)]

fig, axes = plt.subplots(2, 4, figsize=(12.5, 5.2), sharex=True, sharey=True)
for row, (coef, basis, name) in enumerate([(p_E, basis_E, "E = {1, x, x^2}"),
                                           (p_B, basis_B, "B = {1, 1+x, 1+x+x^2}")]):
    total = np.zeros_like(xs)
    for k, (lbl, fn) in enumerate(basis):
        axes[row, k].plot(xs, coef[k]*fn, color="#0E7C7B")
        axes[row, k].axhline(0, color="gray", lw=0.6)
        axes[row, k].set_title(f"{coef[k]:+g} · {lbl}", fontsize=9)
        total += coef[k]*fn
    axes[row, 3].plot(xs, total, color="black", lw=2)
    axes[row, 3].plot(xs, 1 + xs**2, "--", color="crimson", lw=1.2)
    axes[row, 3].set_title("sum  =  1 + x²", fontsize=9)
    axes[row, 0].set_ylabel(name, fontsize=9)
    print(f"{name:26s} coordinates {np.round(coef, 3)}  max |sum - p| ="
          f" {np.abs(total - (1 + xs**2)).max():.1e}")
plt.tight_layout(); plt.show()


**Exercise 19.** Stay in $P_2$ with the same two bases. (a) Take the calibration curve
$q(x) = 2 - x + 3x^2$: write `q_E` directly from its coefficients, compute `q_B`, and verify by
expanding the combination back out. (b) Is $\{1,\;1-x,\;1-x^2\}$ a basis of $P_2$? Decide it with a
rank or determinant test, and if it is, build its `Gamma_EB`.

> 🤖 *Gemini tip:* "Treat polynomials of degree at most 2 as a 3-D vector space with basis {1, x, x^2}. Show me how to build a change-of-basis matrix to another basis in NumPy, convert a polynomial's coordinates, and test whether three given polynomials form a basis."


In [ ]:
q_E = np.array([2.0, -1.0, 3.0])          # q(x) = 2 - x + 3x^2
cand = np.column_stack([[1.0, 0.0, 0.0], [1.0, -1.0, 0.0], [1.0, 0.0, -1.0]])

# Your code here

In [ ]:
# R^(2x2) in two bases. E: the four unit matrices, coordinates read off as (a, b, c, d).
S1 = np.array([[1.0, 0.0], [0.0, 0.0]])
S2 = np.array([[0.0, 0.0], [0.0, 1.0]])
S3 = np.array([[0.0, 1.0], [1.0, 0.0]])     # symmetric
S4 = np.array([[0.0, 1.0], [-1.0, 0.0]])    # antisymmetric
Gm_EB = np.column_stack([S.flatten() for S in (S1, S2, S3, S4)])
Gm_BE = np.linalg.inv(Gm_EB)
print("Gamma_EB:\n", Gm_EB, "\nGamma_BE:\n", Gm_BE)

M = np.array([[2.0, 3.0], [1.0, 4.0]])      # a 2-D displacement-gradient tensor
M_E = M.flatten()                           # (a, b, c, d)
M_B = Gm_BE @ M_E
print("\nM_E =", M_E, " ->  M_B =", M_B)
print("rebuild:", (M_B[0]*S1 + M_B[1]*S2 + M_B[2]*S3 + M_B[3]*S4).round(4).tolist())
print("symmetric part 0.5(M + M.T):\n", (0.5*(M + M.T)).round(4))
print("-> its coordinates are exactly M_B[:3] on S1, S2, S3;",
      "the rigid rotation is M_B[3] =", M_B[3])


**Exercise 20.** Keep the same two bases of $\mathbb R^{2\times2}$. (a) Verify
`Gamma_EB @ Gamma_BE` is the identity. (b) Take the displacement-gradient tensor
`N = np.array([[0.0, 5.0], [-1.0, 2.0]])`: find `N_E` and `N_B`, and check that the symmetric part you
reconstruct from the first three coordinates of `N_B` equals `0.5*(N + N.T)` computed directly.
(c) Which coordinate of `N_B` would be zero if the tensor were purely a strain with no rotation, and
why does that make physical sense?

> 🤖 *Gemini tip:* "Treat 2x2 matrices as a 4-D vector space. Show me how to build a change-of-basis matrix to the basis of symmetric and antisymmetric matrices, and how the symmetric/antisymmetric split of a matrix appears in those coordinates."


In [ ]:
N = np.array([[0.0, 5.0], [-1.0, 2.0]])

# Your code here

---
## Bonus · Uncovering a hidden image (Holbein's *The Ambassadors*)

Hans Holbein the Younger hid an anamorphic skull in his 1533 painting *The Ambassadors*: distorted by a scaling and a shear so it only resolves into a normal image from an extreme viewing angle — or, here, once we invert the transformation. This is exactly the composition-and-inverse machinery from Sections 9–10 above, applied to a real image: the same recipe you'll use in Module 4 to rectify a tilted scan or a skewed microscope/endoscopy frame.

He applied a vertical squash

$$
HS_1 = \begin{bmatrix} 1 & 0 \\ 0 & 1/8 \end{bmatrix}
$$

then a shear

$$
HS_2 = \begin{bmatrix} 1 & 0 \\ 1/2 & 1 \end{bmatrix}
$$

To undo it we need the inverse of the composition $(HS_2\,HS_1)^{-1}$ — remembering that image coordinates have $y$ pointing *down*, so a $y$-axis flip has to bracket the transform, the same trick as the reflections in Section 11.

In [ ]:
import cv2
import urllib.request

def load_image(url):
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    resp = urllib.request.urlopen(req, timeout=20)
    data = np.asarray(bytearray(resp.read()), dtype="uint8")
    img = cv2.imdecode(data, cv2.IMREAD_COLOR)
    if img is None:
        raise ValueError("Failed to decode image.")
    return img

url = ("https://upload.wikimedia.org/wikipedia/commons/thumb/8/88/"
       "Hans_Holbein_the_Younger_-_The_Ambassadors_-_Google_Art_Project.jpg/"
       "500px-Hans_Holbein_the_Younger_-_The_Ambassadors_-_Google_Art_Project.jpg")
try:
    Ambassadors = load_image(url)
    print("loaded painting, shape:", Ambassadors.shape)
except Exception as e:
    print("could not download the painting here (this cell needs internet -- it works in Colab):", e)
    Ambassadors = np.full((300, 500, 3), 200, dtype=np.uint8)   # placeholder so the notebook still runs
    cv2.putText(Ambassadors, "no internet: placeholder", (20, 150),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 0), 2)

plt.figure(figsize=(5, 5)); plt.imshow(cv2.cvtColor(Ambassadors, cv2.COLOR_BGR2RGB)); plt.axis("off")
plt.title("The Ambassadors (as loaded)"); plt.show()

**Bonus Exercise.** Build the matrix that undoes Holbein's distortion: compose his scaling `HS1` and shear `HS2` in the order he applied them, then invert the composition (remember the `y`-axis flip). Apply the result as an affine warp (`cv2.warpAffine`) to `Ambassadors` and see whether the hidden image resolves.

> 🤖 *Gemini tip:* "I have two 2x2 NumPy matrices representing a scale and a shear that were applied to an image, and I want to build the inverse transform that undoes both, then apply it with cv2.warpAffine. Walk me through composing the matrices, inverting, handling the image y-axis flip, and computing the output canvas size from the transformed corners."

In [ ]:
# Your code here

---
### You now have the geometry
Vectors and the dot product, the norm axioms and the L1/L2/L$\infty$ family, projection,
Gram-Schmidt orthonormalization, matrices as maps, the transformation zoo, orthonormal/rigid motions,
the determinant and rank, change of basis in the $\Gamma_{BE}$ notation — with the difference between
a basis change and a linear map made explicit, and turned into a tool: rotations,
reflections and projections assembled in an easy basis — and the same machinery applied to polynomials and
matrices, plus a bonus look at Holbein's hidden anamorphosis. **Next:** Notebook 2 turns these into
the eigendecomposition and SVD, then least squares and PCA on real biomedical data.
